In [1]:
import numpy as np
import pandas as pd

In [ ]:
## Exercise 4.1
# "Mixture of normals" approach to fat-tailed distribution volatilty model
def simulate_logS_distribution(S, r, q, T, sigma1, sigma2, n):
    '''
    S = initial price
    r = risk-free rate
    q = dividend yield
    T = time to maturity
    sigma1, sigma2 = two volatilites to mimic "mixture of normals" 
    n = number of simulations

    returns array of simulated logS(T) values from "mixture of normals" method
    '''
    logS0 = np.log(S)
    
    # Empty array of size n (change for different number of simulations)
    logST = np.zeros(n)

    for i in range(n):
        x = np.random.choice([0,1]) # randomly choosing from given options
        sigma = x*sigma1 + (1-x)*sigma2
        z = np.random.randn() # could get same thing from np.random.normal(0,1)

        logST[i] = logS0 + (r-q-0.5*sigma**2)*T + sigma*np.sqrt(T)*z

    return logST

In [3]:
S = 100
r = 0.05
q = 0.02
T = 1
sigma1 = 0.2
sigma2 = 0.4
n = 500

logST_sim = simulate_logS_distribution(S, r, q, T, sigma1, sigma2, n)

mean_logST = np.mean(logST_sim)
std_logST = np.std(logST_sim)

twostd_below = (len(logST_sim[logST_sim < (mean_logST - 2*std_logST)]))/len(logST_sim)

print('Mean of logS(T):', mean_logST)
print('Std of logS(T):', std_logST)
print('Two std below:', twostd_below)


Mean of logS(T): 4.597586506225856
Std of logS(T): 0.3275150229888409
Two std below: 0.03


In [ ]:
## Exercise 4.2
# GARCH volatility model
def simulate_GARCH(S, r, q, dt, N, kappa, Lambda, theta, sigma0, n):
    """ 
    S = initial price
    r = risk-free rate
    q = dividend yield
    dt = time increment
    N = number of time periods
    kappa = reversion rate perameter
    Lambda = volatility weight
    theta = long-run variance
    sigma0 = initial volatility
    n = number of simulations

    returns array of simulated logS(T) values using GARCH
    """

    logST = []

    # GARCH constants
    a = kappa * theta
    b = (1 - kappa) * (1 - Lambda)
    c = (1 - kappa) * Lambda

    for i in range(n):
        LogS = np.log(S)
        sigma = sigma0

        for j in range(N):
            y = sigma * np.random.randn() # could get same thing from np.random.normal(0,1)
            LogS += (r - q - 0.5 * sigma**2) * dt + np.sqrt(dt) * y # add random path to initial LogS
            sigma = np.sqrt(a + b * y**2 + c * sigma**2) # update volatility

        logST.append(LogS)

    return np.array(logST)

In [11]:
S = 100
r = 0.05
q = 0.02
N = 252 # assuming 252 trading days/year
dt = 1/N
sigma0 = 0.3
theta = 0.09
n = 500

# Trying a few values for kappa and Lambda to find fat tails
kappas = [0.1, 0.2, 0.3]
Lambdas = [0.1, 0.5, 0.9]

for k in kappas:
    for L in Lambdas:
        logST = simulate_GARCH(S, r, q, dt, N, k, L, theta, sigma0, n)
        mean_logST = np.mean(logST)
        std_logST = np.std(logST)
        fraction_below = np.sum(logST < (mean_logST - 2*std_logST)) / len(logST)

        print('Kappa:', k, 'Lambda:', L)
        print('Mean logS(T):', mean_logST)
        print('2sigma below:', fraction_below)

Kappa: 0.1 Lambda: 0.1
Mean logS(T): 4.606262939754069
2sigma below: 0.02
Kappa: 0.1 Lambda: 0.5
Mean logS(T): 4.6199963328450355
2sigma below: 0.018
Kappa: 0.1 Lambda: 0.9
Mean logS(T): 4.571569697400626
2sigma below: 0.026
Kappa: 0.2 Lambda: 0.1
Mean logS(T): 4.605738186690917
2sigma below: 0.024
Kappa: 0.2 Lambda: 0.5
Mean logS(T): 4.580359302321019
2sigma below: 0.022
Kappa: 0.2 Lambda: 0.9
Mean logS(T): 4.596834368208218
2sigma below: 0.024
Kappa: 0.3 Lambda: 0.1
Mean logS(T): 4.582318143111381
2sigma below: 0.018
Kappa: 0.3 Lambda: 0.5
Mean logS(T): 4.573330335749172
2sigma below: 0.032
Kappa: 0.3 Lambda: 0.9
Mean logS(T): 4.588773417825339
2sigma below: 0.012


In [ ]:
## Exercise 4.3
# Heston stochastic volatility model
def simulate_stochastic_volatility(S, r, q, dt, N, kappa, theta, gamma, rho, sigma0, n):
    """ 
    S = initial price
    r = risk-free rate
    q = dividend yield
    dt = time increment
    N = number of time periods
    kappa = reversion rate perameter
    gamma = change in delta with S
    rho = correlation between z and z*
    sigma0 = initial volatility
    n = number of simulations

    returns array of simulated logS(T) values using Heston stochastic volatility
    """

    logST = []

    for i in range(n):
        LogS = np.log(S)
        var = sigma0**2
        sigma = sigma0

        for j in range(N):
            z1 = np.random.randn()
            z2 = np.random.randn()
            zStar = rho * z1 + np.sqrt(1 - rho**2) * z2

            LogS += (r - q - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*z1

            logST.append(LogS)

    return logST

In [15]:
S = 100
r = 0.05
q = 0.02
N = 252 # assuming 252 trading days/year
dt = 1/N
sigma0 = 0.3
theta = 0.09
n = 500

# Trying a few values for kappa, gamma, and rho to find fat tails
kappas = [0.1, 0.2, 0.3]
gammas = [0.1, 0.5, 0.9]
rhos = [-0.9, -0.5, 0.0]

max_fraction_below = 0.0

for k in kappas:
    for g in gammas:
        for p in rhos:
            logST = simulate_stochastic_volatility(S, r, q, dt, N, k, theta, g, p, sigma0, n)
            mean_logST = np.mean(logST)
            std_logST = np.std(logST)
            fraction_below = np.sum(logST < (mean_logST - 2*std_logST)) / len(logST)
            if fraction_below > max_fraction_below:
                max_fraction_below = fraction_below
                maxes = [k, g, p]


            print('Kappa:', k, 'Gamma:', g, 'Rho:', p)
            print('Mean logS(T):', mean_logST)
            print('2sigma below:', fraction_below)

Kappa: 0.1 Gamma: 0.1 Rho: -0.9
Mean logS(T): 4.595210060660194
2sigma below: 0.026452380952380953
Kappa: 0.1 Gamma: 0.1 Rho: -0.5
Mean logS(T): 4.60695782308924
2sigma below: 0.029944444444444444
Kappa: 0.1 Gamma: 0.1 Rho: 0.0
Mean logS(T): 4.610151605249388
2sigma below: 0.02738095238095238
Kappa: 0.1 Gamma: 0.5 Rho: -0.9
Mean logS(T): 4.607118537013523
2sigma below: 0.026404761904761903
Kappa: 0.1 Gamma: 0.5 Rho: -0.5
Mean logS(T): 4.601337236030371
2sigma below: 0.028023809523809524
Kappa: 0.1 Gamma: 0.5 Rho: 0.0
Mean logS(T): 4.581859332174704
2sigma below: 0.030206349206349205
Kappa: 0.1 Gamma: 0.9 Rho: -0.9
Mean logS(T): 4.607421030880023
2sigma below: 0.02426984126984127
Kappa: 0.1 Gamma: 0.9 Rho: -0.5
Mean logS(T): 4.618711570071262
2sigma below: 0.025103174603174603
Kappa: 0.1 Gamma: 0.9 Rho: 0.0
Mean logS(T): 4.607342061034968
2sigma below: 0.026944444444444444
Kappa: 0.2 Gamma: 0.1 Rho: -0.9
Mean logS(T): 4.597828054126021
2sigma below: 0.02976190476190476
Kappa: 0.2 Gamma:

In [16]:
print(max_fraction_below)
print(maxes)

0.03193650793650794
[0.3, 0.5, -0.5]
